## Train BPE Tokenizer on Swissprot Sequences

In [11]:
import os
import json
from tokenizers import Tokenizer, trainers, models

In [12]:
def load_sequences(file_path):
    with open(file_path, 'r') as f:
        for line in f:
            seq = line.strip()

            yield seq

In [13]:
swissprot_sequences_file = '../../data/sequences.txt'

In [24]:
vocab_size = 16384
unk_token = '<unk>'
bos_token = '<bos>'
eos_token = '<eos>'
pad_token = '<pad>'

In [25]:
tokenizer = Tokenizer(model=models.BPE(unk_token=unk_token))
tokenizer.add_special_tokens([bos_token, unk_token, eos_token, pad_token])

tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<bos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"<eos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"<pad>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=None, pre_tokenizer=None, post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[]))

In [26]:
trainer = trainers.BpeTrainer(
    show_progress=True,
    special_tokens=[bos_token, unk_token, eos_token, pad_token],
    min_frequency=5,
    vocab_size=vocab_size,
    max_token_length=6
)

In [27]:
tokenizer.train_from_iterator(load_sequences(swissprot_sequences_file), trainer)

In [28]:
test_seq = 'MYKMYFLKDQKFSLSGTIRINDKTQSEYGSVWCPGLSITGLHHDAIDHNMFEEMETEIIEYLGPWVQAEYRRIKG'

tokenizer.encode(test_seq).tokens

['MY',
 'KM',
 'YFL',
 'KD',
 'QKF',
 'SLSG',
 'TIR',
 'IND',
 'KT',
 'QSE',
 'YGS',
 'VW',
 'CP',
 'GLS',
 'ITGL',
 'HHD',
 'AID',
 'HN',
 'MF',
 'EE',
 'ME',
 'TE',
 'IIE',
 'YLG',
 'PW',
 'VQAE',
 'YRR',
 'IKG']

In [30]:
tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<bos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"<eos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"<pad>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=None, pre_tokenizer=None, post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<bos>":0, "<unk>":1, "<eos>":2, "<pad>":3, "A":4, "B":5, "C":6, "D":7, "E":8, "F":9, "G":10, "H":11, "I":12, "K":13, "L":14, "M":15, "N":16, "O":17, "P":18, "Q":19, "R":20, "S":21, "T":22, "U":23, "V":24, "W":25, "X":26

In [41]:
from transformers import PreTrainedTokenizerFast, AutoTokenizer

In [39]:
wrapped_tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer, unk_token=unk_token, pad_token=pad_token, eos_token=eos_token, bos_token=bos_token)
wrapped_tokenizer

TokenizersBackend(name_or_path='', vocab_size=16384, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [40]:
wrapped_tokenizer.save_pretrained('../../data/eshmun-gpt/tokenizer')

('../../data/eshmun-gpt/tokenizer/tokenizer_config.json',
 '../../data/eshmun-gpt/tokenizer/tokenizer.json')

In [45]:
tok = AutoTokenizer.from_pretrained('../../data/eshmun-gpt/tokenizer')

In [52]:
tok.convert_ids_to_tokens(tok.encode("<bos>" + test_seq))
tok.eos_token_id, tok.pad_token_id

(2, 3)